In [2]:
import tgp
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import scipy
from tgp_adapter import TGPAdapter
from config import PathConfigs
from yield_analysis import analyze_1, analyze_2, show_roi2_tables, show_soi2_tables
import pandas as pd

In [3]:
dirname = Path(PathConfigs.DATA)/"TGP_Test_Run"
data = TGPAdapter(dirname).to_xarray()


In [4]:
data


<xarray.Dataset>
Dimensions:            (cutter_pair_index: 5, B: 9, V: 25, bias: 97)
Coordinates:
  * cutter_pair_index  (cutter_pair_index) int64 0 1 2 3 4
  * B                  (B) float64 0.4 0.425 0.45 0.475 0.5 0.525 0.55 0.575 0.6
  * V                  (V) float64 2.0 2.021 2.042 2.062 ... 2.458 2.479 2.5
  * bias               (bias) float64 -0.134 -0.13 -0.126 ... 0.126 0.13 0.134
Data variables:
    g_ll               (cutter_pair_index, B, V, bias) float64 0.001513 ... 1.28
    g_rr               (cutter_pair_index, B, V, bias) float64 0.004103 ... 0...
    g_lr               (cutter_pair_index, B, V, bias) float64 -1.814e-07 ......
    g_rl               (cutter_pair_index, B, V, bias) float64 -2.637e-07 ......
    L_SI               (cutter_pair_index, B, V) float64 0.0 0.0 0.0 ... 1.0 1.0
    R_SI               (cutter_pair_index, B, V) float64 0.0 0.0 0.0 ... 1.0 1.0
Attributes:
    sample_name:     simulated_1D_nanowire
    surface_charge:  0.0
    disorder_seed:   0

In [5]:
tprep = tgp.prepare.prepare_sim(data, 40.0)

In [20]:
tprep['I_3w_L'].shape

(5, 9, 25)

In [6]:
T_mk = 30
res = [analyze_1(data, T_mK = 30)]


In [7]:
df_1 = pd.DataFrame(res)
df_1_passed = df_1[~df_1.V_min.isna()]

In [8]:
# Print all the bounding boxes
cols = [
    "sample_name",
    "disorder_seed",
    "surface_charge",
    "B_min",
    "V_min",
    "B_max",
    "V_max",
]
df_1_passed[cols]

,sample_name,disorder_seed,surface_charge,B_min,V_min,B_max,V_max
0,simulated_1D_nanowire,0,0.0,0.4,2.033333,0.6,2.5


In [9]:
# Print the number of passed devices
passed_devices = df_1_passed.groupby(["sample_name", "surface_charge"]).apply(len)
total = df_1.groupby(["sample_name", "surface_charge"]).apply(len)
df = pd.concat([passed_devices, total], axis=1)
df.columns = ["# passed", "total"]
df

,,# passed,total
sample_name,surface_charge,,
simulated_1D_nanowire,0.0,1,1


In [11]:
results = [analyze_2(data, T_mK = 40, B_max = 0.6)]

In [11]:
df_stats = pd.DataFrame(results)
show_roi2_tables(df_stats)

,,true positive,false positive,not passed,confidence interval_low,confidence interval_high
sample_name,surface_charge,,,,,
simulated_1D_nanowire,0.0,0.0,0.0,0.0,0.0,100.0


,,false positive,not passed,total,true positive
simulated_1D_nanowire,0.0,0.00,1.00,1.00,0.00
